# Job Radar Automático — Data Analyst (InHire ATS)

Este notebook monitora vagas em empresas que usam o ATS **InHire**.

Objetivo:

- Encontrar vagas de **Data Analyst / Analista de Dados / Business Analyst**
- Filtrar vagas **Remotas**
- Exibir resultados em um **DataFrame Pandas organizado**
- Preparar integração futura com **alertas no Telegram**

Autor: Mario Schenkel

Este projeto pode ser expandido para monitorar centenas de empresas automaticamente.

# 1️⃣ Instalação de bibliotecas

Execute apenas na primeira vez.

In [ ]:
!pip install requests pandas tqdm

# 2️⃣ Imports

In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
from datetime import datetime

pd.set_option('display.max_colwidth', None)

# 3️⃣ Configurações do Radar

Aqui você pode alterar:

- empresas monitoradas
- termos de busca
- termos para detectar trabalho remoto

Isso facilita futuras customizações.

In [ ]:
COMPANIES = [
    "sympla",
    "kiwify",
    "contabilizei",
    "alura",
    "kpmg"
]

DATA_KEYWORDS = [
    "data analyst",
    "analista de dados",
    "business analyst",
    "analista de negócios",
    "analytics",
    "bi analyst"
]

REMOTE_KEYWORDS = [
    "remote",
    "remoto",
    "anywhere",
    "home office"
]

# 4️⃣ Funções de filtro

Estas funções verificam se a vaga corresponde ao perfil desejado.

In [ ]:
def is_data_job(title):
    if not title:
        return False

    title = title.lower()

    return any(keyword in title for keyword in DATA_KEYWORDS)


def is_remote(location):
    if not location:
        return False

    location = location.lower()

    return any(keyword in location for keyword in REMOTE_KEYWORDS)

# 5️⃣ Função para buscar vagas via API InHire

Endpoint utilizado:

https://api.inhire.app/job-posts/public/pages

Este endpoint permite paginação.

In [ ]:
def fetch_jobs(company, max_pages=3):

    jobs = []

    for page in range(max_pages):

        url = f"https://api.inhire.app/job-posts/public/pages?page={page}&size=50&tenant={company}"

        try:

            response = requests.get(url, timeout=10)

            if response.status_code != 200:
                continue

            data = response.json()

            if "content" not in data:
                break

            for job in data["content"]:

                jobs.append({
                    "Empresa": company,
                    "Titulo": job.get("title"),
                    "Local": job.get("location"),
                    "DataPublicacao": job.get("createdAt"),
                    "Link": f"https://{company}.inhire.app/vagas/{job.get('slug')}"
                })

        except Exception as e:
            print("Erro ao buscar", company, e)

    return jobs

# 6️⃣ Executar o Radar

Este passo consulta todas as empresas configuradas.

In [ ]:
results = []

for company in tqdm(COMPANIES):

    jobs = fetch_jobs(company)

    results.extend(jobs)

df = pd.DataFrame(results)

print("Total de vagas coletadas:", len(df))

df.head()

# 7️⃣ Aplicar filtros Data + Remote

In [ ]:
df = df[df["Titulo"].apply(is_data_job)]

df = df[df["Local"].apply(is_remote)]

df

# 8️⃣ Ordenar por data

In [ ]:
df["DataPublicacao"] = pd.to_datetime(df["DataPublicacao"], errors="coerce")

df = df.sort_values("DataPublicacao", ascending=False)

df

# 9️⃣ (Opcional) Envio para Telegram

Configure abaixo se quiser receber alertas automáticos.

In [ ]:
TELEGRAM_BOT_TOKEN = "SEU_TOKEN_AQUI"
TELEGRAM_CHAT_ID = "SEU_CHAT_ID_AQUI"

# Função de envio

In [ ]:
def send_telegram(message):

    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"

    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message
    }

    requests.post(url, data=payload)

# Enviar vagas encontradas

In [ ]:
for _, row in df.iterrows():

    message = f"🚨 Nova vaga encontrada\n\nEmpresa: {row['Empresa']}\nCargo: {row['Titulo']}\nLink: {row['Link']}"

    print(message)

    # descomente para ativar envio
    # send_telegram(message)